# EduPro: Learner Demographics & Course Enrollment Behavior Analysis

This notebook does the full EDA for the project: data integration, demographic segmentation, enrollment distribution, cross-tab / heatmap analysis, KPIs, and behavioral insights. Copy the plots and bullet takeaways from each section straight into your research paper / executive summary.

## 0. Setup
**If you're on Google Colab:** upload `users.csv`, `courses.csv`, `transactions.csv` using the file icon on the left sidebar (Files > Upload), or mount Google Drive if your files live there.

In [ ]:
# Uncomment to mount Google Drive instead of uploading files each time
# from google.colab import drive
# drive.mount('/content/drive')

# Uncomment to use Colab's file upload widget
# from google.colab import files
# uploaded = files.upload()  # then select users.csv, courses.csv, transactions.csv

!pip install -q pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)

# Update these paths to wherever your files are (e.g. '/content/drive/MyDrive/EduPro/users.csv')
users = pd.read_csv('users.csv')
courses = pd.read_csv('courses.csv')
transactions = pd.read_csv('transactions.csv')

transactions['TransactionDate'] = pd.to_datetime(transactions['TransactionDate'])

print(users.shape, courses.shape, transactions.shape)
users.head()

## 1. Data Integration
Join Users ↔ Transactions ↔ Courses on `UserID` and `CourseID`, and check referential integrity (any transactions pointing to a UserID or CourseID that doesn't exist).

In [ ]:
# Referential integrity checks
orphan_users = transactions[~transactions['UserID'].isin(users['UserID'])]
orphan_courses = transactions[~transactions['CourseID'].isin(courses['CourseID'])]
print(f"Transactions with unknown UserID: {len(orphan_users)}")
print(f"Transactions with unknown CourseID: {len(orphan_courses)}")

# Full merged dataset (one row per enrollment/transaction)
df = (transactions
      .merge(users, on='UserID', how='left')
      .merge(courses, on='CourseID', how='left'))

# Age bands used throughout the analysis
bins = [0, 17, 25, 35, 45, 200]
labels = ['<18', '18-25', '26-35', '36-45', '45+']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
users['AgeGroup'] = pd.cut(users['Age'], bins=bins, labels=labels)

df.head()

## 2. Learner Demographic Analysis

In [ ]:
# Age distribution of the LEARNER BASE (unique users, not enrollments)
fig, ax = plt.subplots()
sns.histplot(users['Age'], bins=20, kde=True, ax=ax)
ax.set_title('Age Distribution of Learners')
plt.show()

age_group_counts = users['AgeGroup'].value_counts().reindex(labels)
print(age_group_counts)

In [ ]:
# Gender distribution
gender_counts = users['Gender'].value_counts()
fig, ax = plt.subplots()
gender_counts.plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452','#55A868'])
ax.set_title('Gender Distribution of Learners')
ax.set_ylabel('Number of Users')
plt.show()

print((gender_counts / gender_counts.sum() * 100).round(1).astype(str) + '%')

## 3. Enrollment Distribution Analysis

In [ ]:
# Enrollments by age group (note: this counts ENROLLMENTS, not unique users)
fig, ax = plt.subplots()
df['AgeGroup'].value_counts().reindex(labels).plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_title('Enrollments by Age Group')
ax.set_ylabel('Number of Enrollments')
plt.show()

In [ ]:
for col, title in [('CourseCategory','Course Category'), ('CourseType','Course Type'), ('CourseLevel','Course Level')]:
    fig, ax = plt.subplots()
    df[col].value_counts().plot(kind='bar', ax=ax, color='#55A868')
    ax.set_title(f'Enrollments by {title}')
    ax.set_ylabel('Enrollments')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

print('Most popular category:', df['CourseCategory'].value_counts().idxmax())
print('Least popular category:', df['CourseCategory'].value_counts().idxmin())

## 4. Demographics x Course Preference Analysis

In [ ]:
# Heatmap: Age group vs Course Category
pivot_age_cat = pd.crosstab(df['AgeGroup'], df['CourseCategory'])
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(pivot_age_cat, annot=True, fmt='d', cmap='YlGnBu', ax=ax)
ax.set_title('Age Group vs Course Category (Enrollment Counts)')
plt.tight_layout()
plt.show()

In [ ]:
# Gender vs Course Level comparison
pivot_gender_level = pd.crosstab(df['Gender'], df['CourseLevel'], normalize='index') * 100
fig, ax = plt.subplots()
pivot_gender_level.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('Course Level Preference by Gender (%)')
ax.set_ylabel('% of enrollments')
plt.legend(title='Course Level', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

pivot_gender_level.round(1)

## 5. Behavioral Insights

In [ ]:
enrollments_per_user = df.groupby('UserID').size()

print('Average courses taken per learner:', round(enrollments_per_user.mean(), 2))
print('Median courses taken per learner:', enrollments_per_user.median())

# Enrollment concentration: what % of enrollments come from the top 10% most active users
sorted_counts = enrollments_per_user.sort_values(ascending=False)
top_10pct_n = max(1, int(len(sorted_counts) * 0.10))
concentration = sorted_counts.head(top_10pct_n).sum() / sorted_counts.sum() * 100
print(f'Top 10% most active users account for {concentration:.1f}% of all enrollments')

fig, ax = plt.subplots()
sns.histplot(enrollments_per_user, bins=20, ax=ax)
ax.set_title('Distribution of Enrollments per Learner')
ax.set_xlabel('Number of courses enrolled')
plt.show()

In [ ]:
# Beginner vs Advanced learner behavior: which categories do beginners favor vs advanced learners?
level_cat = pd.crosstab(df['CourseLevel'], df['CourseCategory'], normalize='index') * 100
level_cat.round(1)

## 6. KPI Summary Table
Pulls together the KPIs called for in the project brief into one table you can paste into the executive summary.

In [ ]:
kpis = {
    'Total Enrollments': len(df),
    'Total Unique Learners': users['UserID'].nunique(),
    'Total Courses Offered': courses['CourseID'].nunique(),
    'Avg Enrollments per Learner': round(enrollments_per_user.mean(), 2),
    'Most Popular Category': df['CourseCategory'].value_counts().idxmax(),
    'Least Popular Category': df['CourseCategory'].value_counts().idxmin(),
    'Most Popular Age Group (by enrollments)': df['AgeGroup'].value_counts().idxmax(),
    'Most Common Course Level': df['CourseLevel'].value_counts().idxmax(),
    'Gender Split (%)': (users['Gender'].value_counts(normalize=True) * 100).round(1).to_dict()
}

for k, v in kpis.items():
    print(f'{k}: {v}')

## 7. Auto-Draft: Insight Bullets for the Research Paper
This just formats the numbers above into ready-to-edit sentences — review and refine the wording before pasting into your report.

In [ ]:
top_age_group = df['AgeGroup'].value_counts().idxmax()
top_category = df['CourseCategory'].value_counts().idxmax()
top_level = df['CourseLevel'].value_counts().idxmax()

insights = [
    f"The learner base is concentrated in the {top_age_group} age band, which also drives the largest share of enrollments.",
    f"'{top_category}' is the most in-demand course category on the platform, suggesting where to prioritize new content investment.",
    f"'{top_level}'-level courses account for the largest share of enrollments, indicating most learners are still building foundational skills.",
    f"The top 10% most active learners drive {concentration:.1f}% of total enrollments — a strong argument for retention-focused engagement programs aimed at the long tail of casual users.",
    "Gender-based differences in course-level preference (see Section 4) can inform more targeted, inclusive marketing messaging."
]

for i, s in enumerate(insights, 1):
    print(f"{i}. {s}")